In [ ]:
import mlflow
from datetime import datetime
import json
import tempfile

In [ ]:
# Set the MLflow experiment (creates if not exists) — use a shared path for team visibility
# In CI/CD, this groups all runs (local, PR-triggered, prod) under one experiment
mlflow.set_experiment("/Shared/LSEG-Ingestion-Experiments")  # Customize name/path as needed

In [ ]:
# Start an MLflow run — wraps the entire ingestion process
# Run name includes timestamp + test_folder for easy identification in UI
test_folder = dbutils.widgets.get("test_folder")
run_name = f"Ingestion-{test_folder}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
with mlflow.start_run(run_name=run_name) as active_run:
    # Log key parameters (from widgets/config) — great for CI/CD reproducibility
    mlflow.log_param("config_path", config_path)
    mlflow.log_param("test_folder", test_folder)
    mlflow.log_param("run_mode", run_mode)
    mlflow.log_param("triggered_by", triggered_by)  # e.g., "github-actions" in CI/CD
    mlflow.log_param("pipeline_name", cfg["pipeline_name"])  # From your YAML
    mlflow.log_param("source_name", cfg["source_name"])
    mlflow.log_param("dataset_name", cfg["dataset_name"])

    # --- Existing code: Load YAML config, print loaded values, etc. ---
    # (Insert your existing config loading and printing code here)

    # --- Existing code: Build schema from YAML ---
    # (Insert your schema building code here)

    # --- Existing code: Read stream with Autoloader ---
    # (Insert your stream reading code here, e.g., df = spark.readStream...)

    # --- After reading the data (e.g., after df = spark.readStream...), log initial metrics ---
    # For demo: Count rows if in batch mode (availableNow) — adjust based on your stream
    # In streaming, you might need to use foreachBatch to compute/log metrics per micro-batch
    if "df" in locals():  # Assuming 'df' is your DataFrame
        rows_read = df.count() if not df.isStreaming else 0  # For streaming, log per batch later
        mlflow.log_metric("rows_read", rows_read)

    # --- Existing code: DQ checks (missing rows, nulls, etc.) ---
    # (Insert your DQ logic here)
    # After DQ: Log DQ metrics — e.g., assume you have variables like dq_failures, null_rate
    # mlflow.log_metric("dq_failures", dq_failures)  # Replace with your actual DQ vars
    # mlflow.log_metric("null_rate_close_price", null_rate_close_price)  # Example

    # --- NEW: Generate and log a simple DQ report as artifact ---
    # Create a JSON report (or CSV) with DQ results — upload as artifact
    dq_report = {
        "expected_min_rows": expected_min_rows,
        "actual_rows": rows_read,  # Replace with actual
        "required_columns": required_cols,
        "null_thresholds": null_thresholds,
        # Add your policy results, e.g., "missing_rows_severity": "ERROR"
    }
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as temp_file:
        json.dump(dq_report, temp_file)
        temp_file_path = temp_file.name
    mlflow.log_artifact(temp_file_path, artifact_path="dq_reports")  # Logs to MLflow

    # --- NEW: Log the YAML config as an artifact ---
    # Copy YAML to temp file and log
    yaml_content = dbutils.fs.head(config_path, 100000)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".yml", delete=False) as temp_yaml:
        temp_yaml.write(yaml_content)
        temp_yaml_path = temp_yaml.name
    mlflow.log_artifact(temp_yaml_path, artifact_path="configs")

    # --- Existing code: Write stream to raw table ---
    # (Insert your writeStream code here)

    # --- After write (if batch), log final metrics ---
    # e.g., rows_written = spark.table(raw_table).count()
    # mlflow.log_metric("rows_written", rows_written)

    # --- Existing code: Any post-write checks or prints ---

# --- End of notebook: MLflow run auto-ends with the 'with' block ---
